In [1]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
import cv2
import mediapipe as mp
import winsound
import datetime
import random
import time
import pandas as pd

# =============================
# Vision Agent - Improved with Scale Invariance
# =============================
class VisionAgent:
    def __init__(self):
        self.mp_face_mesh = mp.solutions.face_mesh
        self.face_mesh = self.mp_face_mesh.FaceMesh(
            max_num_faces=2,
            refine_landmarks=True,
            min_detection_confidence=0.7,
            min_tracking_confidence=0.7
        )

    def perceive(self, frame):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.face_mesh.process(rgb)
        face_count = 0
        head_direction = "Straight"

        if results.multi_face_landmarks:
            face_count = len(results.multi_face_landmarks)
            for face_landmarks in results.multi_face_landmarks:
                h, w, _ = frame.shape
                nose = face_landmarks.landmark[1]
                left_cheek = face_landmarks.landmark[234]
                right_cheek = face_landmarks.landmark[454]

                # Scale Invariant Logic: Using 20% of face width as threshold
                face_width = abs(right_cheek.x - left_cheek.x)
                nose_offset = nose.x - (left_cheek.x + right_cheek.x) / 2
                threshold = face_width * 0.2 

                if nose_offset > threshold:
                    head_direction = "Looking Left"
                elif nose_offset < -threshold:
                    head_direction = "Looking Right"

        return face_count, head_direction

# =============================
# Generative AI Agent (RE-DEFINED HERE)
# =============================
class GenerativeAgent:
    def generate_reason(self, status):
        explanations = {
            "Face Missing": [
                "Candidate left camera view. Possible attempt to consult external resources.",
                "Face not visible. This may indicate suspicious movement."
            ],
            "Multiple Faces": [
                "More than one face detected. This suggests external assistance.",
                "Unauthorized person detected in camera frame."
            ],
            "Looking Left": [
                "Repeated left glance detected. Possible off-screen reference.",
                "Head turned left. May indicate distraction or cheating."
            ],
            "Looking Right": [
                "Repeated right glance detected. Possible off-screen help.",
                "Head turned right. Suspicious behavior observed."
            ]
        }
        if status in explanations:
            return random.choice(explanations[status])
        return "Candidate behavior appears normal."

    def assess_risk(self, warnings):
        if warnings == 1: return "Low Risk"
        elif warnings == 2: return "Medium Risk"
        elif warnings >= 3: return "High Risk"
        return "Safe"

# =============================
# Action Agent - Improved with Temporal Buffer & Logging
# =============================
class ActionAgent:
    def __init__(self, max_warnings=3):
        self.warnings = 0
        self.previous_status = "Normal"
        self.max_warnings = max_warnings
        self.violation_start_time = None
        self.BUFFER_DURATION = 1.5  # Ignore movements shorter than 1.5s
        self.event_log = []

    def act(self, status):
        if status != "Normal":
            if self.violation_start_time is None:
                self.violation_start_time = time.time()
            
            elapsed = time.time() - self.violation_start_time
            
            # Action logic: Only warn if behavior is sustained
            if elapsed >= self.BUFFER_DURATION and status != self.previous_status:
                self.warnings += 1
                winsound.Beep(2000, 300)
                self.previous_status = status
                self.event_log.append({
                    "time": datetime.datetime.now().strftime("%H:%M:%S"),
                    "violation": status,
                    "warning_count": self.warnings
                })
                self.violation_start_time = None 
        else:
            self.violation_start_time = None
            self.previous_status = "Normal"
        return self.warnings

    def save_log(self):
        if self.event_log:
            df = pd.DataFrame(self.event_log)
            df.to_csv("proctoring_evidence.csv", index=False)
            print("Log saved: proctoring_evidence.csv")

# =============================
# Main Execution Loop
# =============================
vision = VisionAgent()
gen_ai = GenerativeAgent()
action = ActionAgent(max_warnings=3)

video = cv2.VideoCapture(0)

while video.isOpened():
    success, frame = video.read()
    if not success: break

    face_count, head_direction = vision.perceive(frame)
    
    if face_count == 0: status = "Face Missing"
    elif face_count > 1: status = "Multiple Faces"
    elif head_direction != "Straight": status = head_direction
    else: status = "Normal"

    warnings = action.act(status)
    risk_level = gen_ai.assess_risk(warnings)
    explanation = gen_ai.generate_reason(status)

    # UI Overlay
    cv2.putText(frame, f"Status: {status}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.putText(frame, f"Risk: {risk_level}", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 165, 255), 2)
    cv2.putText(frame, f"Warnings: {warnings}/3", (20, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    cv2.putText(frame, explanation[:70], (20, 160), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

    if action.violation_start_time:
        timer = int(time.time() - action.violation_start_time)
        cv2.putText(frame, f"Analyzing Intent: {timer}s", (20, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

    if warnings >= 3:
        cv2.putText(frame, "EXAM TERMINATED", (120, 240), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 4)
        cv2.imshow("Agentic Proctor Demo", frame)
        cv2.waitKey(3000)
        break

    cv2.imshow("Agentic Proctor Demo", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

action.save_log()
video.release()
cv2.destroyAllWindows()

import pandas as pd

# Load the generated log
df = pd.read_csv("proctoring_evidence.csv")

# Display the data
display(df)

Log saved: proctoring_evidence.csv


,time,violation,warning_count
0,22:59:01,Looking Left,1
1,22:59:10,Looking Right,2
2,22:59:16,Face Missing,3
